In [ ]:
#download necessary libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import statsmodels.api as sm
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import ttest_ind
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats

# Exploring the Dataset

For this project, we are working with the stroke prediction dataset (healthcare-dataset-stroke-data.csv) available on Kaggle https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset?resource=download. The dataset includes 5,110 records and 12 attributes describing patient demographics, health conditions, and stroke outcomes.

The variables are as follows:

id – unique patient identifier

gender – patient’s gender: Male, Female, or Other

age – patient’s age in years

hypertension – 0 if the patient does not have hypertension, 1 if diagnosed with hypertension

heart_disease – 0 if no history of heart disease, 1 if heart disease is present

ever_married – marital status: Yes or No

work_type – type of employment: children, Govt_job, Never_worked, Private, or Self-employed

Residence_type – living area: Rural or Urban

avg_glucose_level – average blood glucose concentration

bmi – body mass index

smoking_status – smoking history: formerly smoked, never smoked, smokes, or Unknown

stroke – outcome variable: 1 if the patient experienced a stroke, 0 otherwise


**********************************************************
Our purpose is to examine whether age plays a significant role in the occurrence of stroke, as age is widely recognized in healthcare settings as a critical risk factor for many diseases. Specifically, we aim to test the hypothesis:

Hypothesis: Older age increases the odds of stroke.

H₀ (Null Hypothesis): Age has no effect on the likelihood of stroke.

H₁ (Alternative Hypothesis): Older age increases the odds of stroke.

Validating this relationship can help health institutes and practitioners better identify high-risk populations, take preventive measures, and ultimately improve outcomes for patients vulnerable to stroke.

In [ ]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")
df.head()

we see from the data that bmi has some missing values let's take a look at the distribution of BMI

In [ ]:

color = ['mediumpurple', 'gold']
sns.histplot(data=df, x='bmi', bins=30, kde=True, hue='stroke', palette= color, multiple='stack')
plt.title('BMI by Stroke')
plt.xlabel('BMI')
plt.ylabel('Frequency')
plt.legend(title='Stroke', labels=['No', 'Yes'])
plt.show()

The plot indicates that stroke occurrence is most common among individuals with a BMI between 20 and 40.

#  Data Cleaning & visualizations
Perform exploratory data analysis (EDA) on the dataset to understand its characteristics, identify potential issues, and gain insights for further analysis or modeling.

In [ ]:
df.info()

df.isna().sum()

In the dataset, approximately 4% of BMI values were missing. To prevent data leakage, no imputation or removal will be performed during this stage. After splitting the dataset into training and test subsets, mean imputation will be applied to the BMI variable using statistics derived solely from the training data.

In [ ]:
df.duplicated().sum()

Now we are going to see the barplot and a  pie chart of our target  variable

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x='stroke', palette=color)
plt.title('Stroke Class Distribution')
plt.xlabel('Stroke')
plt.ylabel('Count')
plt.xticks([0, 1], ['No', 'Yes'])
plt.tight_layout()
plt.show()

In [ ]:
df["stroke"].value_counts()

We observe that 95.1% of patients did not have a stroke while only 4.9% did. This shows that our target variable is highly imbalanced. Because of this, we will not rely on accuracy alone, as a model could achieve high accuracy by always predicting ‘No Stroke.’ Instead, we will evaluate our models using metrics such as Recall, F1-score, and ROC AUC, which are better suited for imbalanced datasets.


And then we are going to look at the distributions of numerical variables and then the object variables.

In [ ]:
continuous_vars = ["age", "avg_glucose_level", "bmi"]

plt.figure(figsize=(15, 5))

for i, col in enumerate(continuous_vars, 1):
    plt.subplot(1, 3, i)
    sns.histplot(df[col], bins=20, color='teal', stat='density', alpha=0.6)
    sns.kdeplot(df[col], color='orange', lw=2)
    plt.title(f"Distribution of {col}")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

for i, col in enumerate(continuous_vars, 1):
    plt.subplot(1, 3, i)
    sns.boxplot(x='stroke', y=col, data=df, palette='Set2')
    plt.title(f"{col} by Stroke")
    
plt.tight_layout()
plt.show()

The exploratory plots show that age is widely distributed across the dataset, with a larger concentration in middle to older age groups. Average glucose level is heavily right-skewed, with most patients at lower levels but a noticeable spread into very high values. BMI also shows a right-skewed pattern, clustering mostly between 20 and 40. When broken down by stroke occurrence, patients with strokes tend to be older on average, glucose levels appear more variable among stroke cases, and BMI distributions overlap substantially between groups.

In [ ]:
fig1, ax1 = plt.subplots(4, 2, figsize=(13, 16))

cols = ['skyblue', 'purple']
objects = ['gender', 'ever_married', 'work_type', 'Residence_type', 
           "smoking_status", "hypertension", "heart_disease"]

ax1 = ax1.flatten()

for i, column in enumerate(objects):
    sns.countplot(data=df, x=column, hue='stroke', palette=cols, ax=ax1[i])

if len(objects) < len(ax1):
    fig1.delaxes(ax1[len(objects)])

plt.tight_layout()
plt.show()

Since the (Other) category under gender does not add value to the analysis, we will exclude it from the data.

In [ ]:
df.gender.value_counts()

In [ ]:
drp = df[df.gender == 'Other'].index
df.drop(drp , inplace=True)

In [ ]:
plt.figure(figsize=(6, 4))
colors = ['skyblue', 'purple']
sns.countplot(df, x='gender', hue='stroke', palette=colors)
plt.xlabel('Gender')
plt.ylabel('count')
plt.title('Gender Distribution')
plt.show()

We will drop the id attribute, as it has no impact on the analysis.

In [ ]:
removed_col = df.columns.difference(['id'])
df = df[removed_col]
print(df.shape)
df.head()

Although the dataset is clean and free of null values, the smoking_status column still contains some entries labeled as (Unknown).

In [ ]:
df["smoking_status"]

Therefore, we will restrict the smoking_status column to only three categories: never smoked, formerly smoked, and smokes, as shown below.

In [ ]:
df = df.loc[(df.smoking_status == "never smoked") |(df.smoking_status == "smokes") 
| (df.smoking_status == "formerly smoked") ]

df.smoking_status.value_counts()

Next, we examine the correlation between the numerical features and the target variable.

In [ ]:
print("Correlation Between numerical Features and Stroke:")
features = ['avg_glucose_level', 'bmi', 'age']
target = 'stroke'
correlation = df[features + [target]].corr()

print(correlation)

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(correlation, cmap='YlGnBu', fmt=".2f", linewidths=.5, annot=True)
plt.show()

In the cleaned dataset, the correlation heatmap shows that age has the strongest association with stroke (0.25), confirming that older patients are more likely to be at risk. Average glucose level has a weaker positive correlation with stroke (0.13), suggesting that elevated glucose levels may contribute but are less influential than age. BMI shows virtually no correlation with stroke (0.01), indicating that it does not provide meaningful predictive power in this dataset. Among the predictors themselves, age and average glucose level have a mild positive correlation (0.23), while BMI remains only weakly related to the other features.

Now let's look at the Pairplot of the numerical variables

In [ ]:
pairplot_vars = ["age", "avg_glucose_level", "bmi"]

fig = plt.figure(figsize=(12, 10), dpi=100)
sns.pairplot(
    df[pairplot_vars + ["stroke"]],
    vars=pairplot_vars,             
    hue="stroke",
    palette="magma"
)
plt.show()

The pairplot shows how stroke and non-stroke cases are distributed across age, average glucose level, and BMI. Stroke cases appear more concentrated at older ages, supporting the idea that age is a key factor. Patients with higher glucose levels also show more stroke cases, though the separation is less distinct than with age. In contrast, BMI distributions for stroke and non-stroke groups overlap heavily, indicating little distinction between them.

In [ ]:
df.describe()

The summary statistics show that patients range in age from 10 to 82 years, with a mean age of approximately 49, indicating a wide age distribution across the sample. The average glucose level has a mean of about 109 mg/dL, but extends up to 272 mg/dL, suggesting the presence of high-value outliers. Similarly, the mean BMI is 30.3, placing the average individual in the obese range, with extreme values reaching 92, which may represent medical anomalies or recording errors. About 6% of patients have heart disease and 12.5% have hypertension, both recognized risk factors for stroke. The stroke outcome itself is relatively rare, occurring in only 5.7% of cases, creating a notable class imbalance. This imbalance is important to consider in subsequent modeling steps, as it can bias predictive performance if not properly addressed.

# Outlier Detection
In this step, we aim to identify potential outliers in the numerical features of the dataset. Using the interquartile range (IQR) method, we calculate the bounds for each variable to detect values that fall significantly outside the typical range, as these may affect model performance.

In [ ]:
numerical_cols = ['age', 'avg_glucose_level', 'bmi']

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Number of outliers in '{col}': {len(outliers)}")

The outlier analysis shows that age has no extreme values, indicating a consistent distribution, while BMI has 90 outliers and average glucose level has 505 outliers, suggesting skewed distributions with unusually high or low values. These outliers may reflect real medical variation rather than errors, but they could also disproportionately influence model performance, so they should be carefully considered either kept as meaningful clinical data, transformed to reduce skewness, or capped to limit their effect.

# Encode Categorical Data

In [ ]:
#Categorical columns
cat_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

ohe = OneHotEncoder(drop='first', sparse_output=False)

encoded = ohe.fit_transform(df[cat_cols])

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cat_cols), index=df.index)


df_encoded = pd.concat([df.drop(columns=cat_cols), encoded_df], axis=1)

print(df_encoded.shape)
df_encoded.head()


In [ ]:
df_encoded.info()

## Data Analysis Key Findings

The data contains a mix of numerical and categorical features. The numerical variables are age, avg_glucose_level, and bmi, while the categorical variables include gender, ever_married, work_type, Residence_type, and smoking_status. The target variable is stroke (1 = stroke, 0 = no stroke).
We observe that 94.3% of patients did not have a stroke while only 5.7% did, highlighting a strong class imbalance. Because of this, accuracy alone cannot be relied on, as a model could appear to perform well by always predicting “No Stroke.” Instead, model performance will be assessed using Recall, F1-score, and ROC AUC, which are better suited for imbalanced datasets.
Exploratory analysis shows that age is widely distributed, with stroke patients tending to be older. Average glucose level is heavily right-skewed, with many patients at lower values but some extending into very high ranges, while BMI clusters mostly between 20 and 40 but also shows extreme values. When broken down by stroke status, stroke patients are significantly older, have more variable glucose levels, and show substantial overlap in BMI with non-stroke patients. Categorical analysis indicates that ever-married individuals show a slightly higher stroke risk, likely related to age, and strokes are more common among those working in Private and Self-employed sectors.
Correlation analysis confirms that age has the strongest relationship with stroke (0.25), followed by avg_glucose_level with a weaker positive correlation (0.13). BMI shows virtually no correlation (0.01), while gender and Residence Type also demonstrate little direct association. The pairplot further illustrates that stroke cases cluster more heavily in older ages and at higher glucose levels, whereas BMI provides no clear separation.
Summary statistics show that patient ages range from 10 to 82 years, with an average of about 49. The mean glucose level is 109, but values extend to 272, and mean BMI is about 30, in the obese range, with extreme values up to 92. Only 6% of patients have heart disease, 12.5% have hypertension, and stroke occurrence remains rare. Outlier analysis identified 505 glucose outliers and 90 BMI outliers, which may reflect real medical variation but could also distort model performance if left unaddressed.

## Next Step
With the dataset cleaned, explored, and encoded, the next step is to formally test our hypothesis and evaluate predictive performance using machine learning models. We will assess whether age is a significant predictor of stroke through both statistical testing and model-based approaches. On the statistical side, we apply methods such as the Welch’s t-Test with mean difference and 95% confidence intervals for age by stroke status to evaluate group differences. On the modeling side, we use two complementary approaches: Logistic Regression, which provides interpretable coefficients and odds ratios directly linked to inference, and Random Forest, which captures non-linear relationships and highlights variable importance. Together, these methods provide both statistical evidence and predictive validation of the hypothesis.

# Hypothesis:

Older age increases the odds of stroke.

H0 (Null): Age has no effect on the likelihood of stroke.

H1 (Alternative): Older age increases the odds of stroke.

In [ ]:
df_encoded.head()

# Train/Test Split

In [ ]:

X = df_encoded.drop('stroke', axis=1) 
Y = df_encoded['stroke']
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size = 0.3, random_state = 42)

print("X_train: ", X_train.shape)
print("\nX_test: ", X_test.shape )
print("\ny_train: ", y_train.shape )
print("\ny_test: ", y_test.shape)

Now we are going to take care of the missing values in BMI by using SimpleImputer, which is an effective method for handling missing numerical data. It replaces the missing values with a calculated statistic, such as the mean or median, allowing us to retain all observations without dropping rows. This approach ensures that the dataset remains complete for model training while preserving the overall distribution of the BMI variable.

In [ ]:
imputer = SimpleImputer(strategy='mean')
X_train['bmi'] = imputer.fit_transform(X_train[['bmi']])
X_test['bmi'] = imputer.transform(X_test[['bmi']])

In [ ]:
scaler = StandardScaler()

cols = X_train.columns
idx_train = X_train.index
idx_test = X_test.index

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(X_train_scaled, columns=cols, index=idx_train)
X_test = pd.DataFrame(X_test_scaled, columns=cols, index=idx_test)

# Model Selection

#  logistic Regression & Random Forest
We selected Logistic Regression and Random Forest for our analysis. Logistic Regression was chosen because it directly tests our hypothesis, providing interpretable coefficients that quantify how much age affects stroke risk. Random Forest was selected as a complementary model because of its strong predictive performance and ability to rank feature importance. Together, these models allow us to test the hypothesis both statistically and predictively, ensuring robust conclusions.

# 1 logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear')
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
y_proba = log_reg.predict_proba(X_test)[:,1]

#Metrics
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC AUC Score:", round(roc_auc_score(y_test, y_proba),4))

#Odds ratio for Age
cols = X.columns
age_index = list(cols).index('age')
age_coef = log_reg.coef_[0][age_index]
age_or = np.exp(age_coef)

print(f"Age Coefficient = {age_coef:.3f}, Odds Ratio = {age_or:.3f}")

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color='blue', label=f"AUC = {roc_auc_score(y_test, y_proba):.3f}")
plt.plot([0,1], [0,1], linestyle='--', color='gray')  # baseline
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend(loc="lower right")
plt.show()

The logistic regression model achieved an accuracy of 73% with an ROC AUC of 0.79, indicating good overall discriminative ability between stroke and non-stroke cases. The model was more effective at detecting stroke patients (recall = 0.69) than precisely predicting them (precision = 0.15), suggesting it produced more false positives but successfully captured most true stroke cases. The corresponding F1-score of 0.24 reflects this tradeoff, balancing sensitivity and precision. Such a pattern is typical and acceptable in healthcare prediction contexts, where minimizing missed stroke cases is more important than avoiding false alarms. For the non-stroke class, the F1-score of 0.84 demonstrates that the model performs strongly for the majority class while maintaining overall balance. The odds ratio for age (OR = 5.16) confirms that older age is a strong and statistically significant predictor of stroke, with the likelihood of stroke increasing more than fivefold as age rises. Together, these results validate that the model effectively captures age-related risk while highlighting the inherent challenges of class imbalance in medical data.

In [ ]:
constant_cols = [col for col in X_train.columns if X_train[col].nunique() <= 1]
if constant_cols:
    print("Dropping constant cols:", constant_cols)
    X_train = X_train.drop(columns=constant_cols)
    X_test = X_test.drop(columns=constant_cols)


def calculate_vif(df):
    vif = pd.DataFrame()
    vif["feature"] = df.columns
    vif["VIF"] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif

vif_data = calculate_vif(X_train)
high_vif = vif_data[vif_data["VIF"] > 1000]["feature"].tolist()
if high_vif:
    print("Dropping high-VIF cols:", high_vif)
    X_train = X_train.drop(columns=high_vif)
    X_test = X_test.drop(columns=high_vif)


X_train_const = sm.add_constant(X_train, has_constant='add')
X_test_const = sm.add_constant(X_test, has_constant='add')


try:
    logit_model = sm.Logit(y_train, X_train_const)
    result = logit_model.fit(disp=False)
    print(result.summary())

#Odds ratios + 95% CI
    odds_ratios = np.exp(result.params)
    ci = np.exp(result.conf_int())
    ci.columns = ["Lower CI", "Upper CI"]
    print(pd.concat([odds_ratios.rename("OR"), ci], axis=1))

except Exception as e:
    print("Standard Logit failed, switching to regularized:", e)
    result = sm.Logit(y_train, X_train_const).fit_regularized(method='l1', disp=False)
    print(result.summary())

Logistic regression analysis confirmed that age is a highly significant predictor of stroke (p < 0.001). After standardization, the coefficient for age was 1.55 (95% CI: 1.24, 1.87), corresponding to an odds ratio of 4.74 (95% CI: 3.47, 6.5). This indicates that for each one standard deviation increase in age, the odds of stroke increase by more than fourfold.

# 2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=400,class_weight={0:1, 1:10},random_state=42)
rf.fit(X_train, y_train)


y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, y_proba_rf)


threshold = 0.10
y_pred_rf = (y_proba_rf >= threshold).astype(int)

#Metrics
print(f"\nClassification Report (Threshold={threshold}):\n")
print(classification_report(y_test, y_pred_rf, zero_division=0))

print(f"ROC AUC (Random Forest): {auc_rf:.3f}")

In [ ]:
#ROC Curve
fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test, y_proba_rf)
plt.figure(figsize=(6,6))
plt.plot(fpr_rf, tpr_rf, color='teal', label=f"AUC = {roc_auc_score(y_test, y_proba_rf):.2f}")
plt.plot([0,1], [0,1], linestyle='--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest")
plt.legend(loc="lower right")
plt.show()

The Random Forest model achieved an accuracy of 82% with an ROC AUC of 0.77, showing solid overall ability to distinguish between stroke and non-stroke cases. After rebalancing class weights and lowering the prediction threshold to 0.10, the model’s recall for stroke cases improved to 0.42, meaning it successfully identified 42% of actual stroke patients. Precision was 0.15, indicating that while some false positives were introduced, the model became substantially more sensitive to detecting stroke occurrences, an important tradeoff in healthcare applications where missing true cases carries a higher cost than issuing extra alerts.

The corresponding F1-score of 0.22 reflects this balance between recall and precision, while the non-stroke class retained strong performance (F1 = 0.90). Although Random Forest models typically achieve higher AUC scores than logistic regression, the heavily imbalanced nature of this dataset (approximately 5% stroke cases) limits the achievable AUC improvement. Despite this, Random Forest still reinforced the central finding that age remains the dominant risk factor for stroke, contributing the highest feature importance in the model. Overall, the Random Forest provided valuable validation of the logistic regression results, demonstrating consistent predictive patterns under a more flexible, non-linear framework.

In [ ]:
#Feature Importance 
feat_importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top_features = feat_importances.nlargest(10).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
top_features.plot(kind='barh', color='teal')
plt.title("Top 10 Feature Importances - Random Forest", fontsize=13)
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

Random Forest feature importance analysis further confirmed that age is the most influential predictor of stroke, accounting for nearly 35% of the predictive power of the model. This contribution was substantially greater than that of other features such as average glucose level and BMI, which accounted for less than half of the importance of age. Together with the logistic regression results, this reinforces our hypothesis that older age is the strongest risk factor for stroke.

## Welch’s t-Test with Mean Difference and 95% CI for Age by Stroke Status

The purpose of this analysis is to determine whether there is a statistically significant difference in mean age between stroke and non-stroke groups. By applying Welch’s t-test, we account for unequal variances, and by calculating the mean difference with a 95% confidence interval, we quantify both the magnitude and precision of the age difference associated with stroke occurrence.

In [ ]:
#Split groups
stroke_age = df_encoded[df_encoded['stroke'] == 1]['age']
no_stroke_age = df_encoded[df_encoded['stroke'] == 0]['age']

#Welch’s t-test
t_stat, p_val = stats.ttest_ind(stroke_age, no_stroke_age, equal_var=False)
print(f"T-test: t = {t_stat:.3f}, p = {p_val:.3g}\n")

#Mean difference
mean_diff = stroke_age.mean() - no_stroke_age.mean()

#Se of difference
se_diff = np.sqrt(stroke_age.var(ddof=1)/len(stroke_age) + 
                  no_stroke_age.var(ddof=1)/len(no_stroke_age))

df_welch = ( (stroke_age.var(ddof=1)/len(stroke_age) + 
              no_stroke_age.var(ddof=1)/len(no_stroke_age))**2 ) / \
           ( ((stroke_age.var(ddof=1)/len(stroke_age))**2 / (len(stroke_age)-1)) + 
             ((no_stroke_age.var(ddof=1)/len(no_stroke_age))**2 / (len(no_stroke_age)-1)) )

#Critical t value
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha/2, df_welch)

# CI
ci_low = mean_diff - t_crit * se_diff
ci_high = mean_diff + t_crit * se_diff

print(f"Mean difference in age = {mean_diff:.2f}\n")

print(f"95% CI for mean difference = ({ci_low:.2f}, {ci_high:.2f})")


A two-sample t-test showed that stroke patients were significantly older than non-stroke patients (t = 23.1, p < 0.001). On average, stroke patients were 20.5 years older, with a 95% CI of 18.7 to 22.2 years. This provides strong evidence that age is a key risk factor for stroke.


# Summary:

The results from both statistical testing and predictive modeling consistently demonstrate that age is a highly significant predictor of stroke. Logistic regression achieved an accuracy of 73% with an ROC AUC of 0.79, showing strong discriminative ability, and revealed that each one standard deviation increase in age was associated with more than a fourfold increase in the odds of stroke (OR = 4.74, 95% CI: 3.46–6.5, p < 0.001). Random Forest analysis reinforced this finding, identifying age as the most influential predictor, contributing nearly 35% of the model’s predictive power, far exceeding other features such as glucose level and BMI. A Welch’s t-test provided further confirmation, showing stroke patients were on average 20.5 years older than non-stroke patients (t = 23.1, p < 0.001; 95% CI: 18.7–22.2). Taken together, these findings provide overwhelming evidence to reject the null hypothesis and conclude that older age significantly increases the likelihood of stroke, making it the strongest risk factor identified in this dataset.
